In [7]:
"""
covariance_steering.py

Converted from MATLAB -> Python. Uses MOSEK Fusion for solveOMS (mean steering).
Requires:
  pip install numpy scipy mosek        # mosek includes fusion API
  (or follow MOSEK installation guide)

Notes:
- You must provide domain-specific functions:
    - eval_A(x_ref, u_ref, plant)
    - eval_B(x_ref, u_ref, plant)
    - eval_D(x_ref, u_ref, plant)
    - solveOCS_LTV(plant, OCSparams)    # or replace with your LTV solver
  I left clear placeholders where these must be implemented.

- Simulink simulation in the original code is replaced by a simple linear simulation
  (sampling from mvn and integrating with LTV system) as an example. If you have a
  high-fidelity simulator, replace the simulation block.

- solveOMS implemented with MOSEK Fusion (linear dynamics + quadratic objective).
"""

import numpy as np
from scipy.spatial.transform import Rotation as R
from math import sqrt
from mosek.fusion import Model, Matrix, Expr, Domain, ObjectiveSense

In [8]:
# ---------------------------
# Parameters
# ---------------------------
np.random.seed(1)

g = 9.81
DT = 0.01
N = 500
m = 1.0

plant = {}
plant['g'] = g
plant['DT'] = DT
plant['N'] = N
plant['nx'] = 9
plant['nu'] = 3     # initial plant.nu for triple integrator step

# ---------------------------
# OMS params (mean steering)
# ---------------------------
OMSparams = {}
OMSparams['N'] = N
OMSparams['Q'] = 0.0 * np.eye(9)
OMSparams['R'] = 1.0 * np.eye(3)
OMSparams['mu0'] = np.vstack([0.8, 0.8, 0.0, np.zeros((6,1))])[:,0]
OMSparams['muf'] = np.vstack([0.0, -0.5, 0.0, np.zeros((6,1))])[:,0]
# waypoints (MATLAB matrix was 3x4?), in your original you had a 2x4 - check
OMSparams['wp'] = 3.0 * np.array([[1, 1, -1, -1],
                                  [0.5, -0.5, 0.5, -0.5]])
OMSparams['wp_n'] = [100, 200, 300, 400]

In [10]:
# ---------------------------
# Linear triple-integrator matrices
# ---------------------------
plant_lin = plant.copy()
nx = plant['nx']
Ac = np.block([
    [np.zeros((3,3)), np.eye(3), np.zeros((3,3))],
    [np.zeros((3,3)), np.zeros((3,3)), np.eye(3)],
    [np.zeros((3,3)), np.zeros((3,3)), np.zeros((3,3))]
])
Bc = np.vstack([np.zeros((6,3)), np.eye(3)])
A_lin = np.eye(nx) + Ac*DT
B_lin = Bc*DT
plant_lin['A'] = A_lin
plant_lin['B'] = B_lin

In [11]:
def solveOMS_fusion(plant, OMSparams, verbose=False):
    """
    Solve the mean steering (open-loop) optimization with MOSEK Fusion.
    min sum_{k=1..N-1} mu_k' Q mu_k + v_k' R v_k
    s.t. mu_{k+1} = A * mu_k + B * v_k
         mu_1 = mu0, mu_N = muf
         waypoint constraints on positions
    plant.A, plant.B may be time-invariant matrices (for triple integrator),
    or A,B can be sequences (list) for time-varying systems. We accept both:
      - If plant['A'] is a 2D array -> time-invariant
      - If it's a list of arrays length N-1 -> time-varying
    """
    N = OMSparams['N']
    nx = plant['nx']
    try:
        nu = plant['B'].shape[1]
    except Exception:
        # fallback if plant['nu'] exists
        nu = plant.get('nu', OMSparams['R'].shape[0])

    Q = OMSparams['Q']
    R = OMSparams['R']
    mu0 = OMSparams['mu0']
    muf = OMSparams['muf']
    wp = OMSparams['wp']
    wp_n = OMSparams['wp_n']

    # handle time-invariant or time-varying A,B
    A_data = plant['A']
    B_data = plant['B']
    is_timevarying = isinstance(A_data, (list, tuple))

    with Model('solveOMS') as M:
        if not verbose:
            M.setLogHandler(None)

        # variables: mu_k for k=1..N (each nx), v_k for k=1..N-1 (each nu)
        # We'll pack them as separate variables for clarity.
        mu_vars = [ M.variable(f"mu_{k}", nx, Domain.unbounded()) for k in range(N) ]
        v_vars  = [ M.variable(f"v_{k}", nu, Domain.unbounded()) for k in range(N-1) ]

        # dynamics constraints
        for k in range(N-1):
            if is_timevarying:
                A_k = A_data[k]
                B_k = B_data[k]
            else:
                A_k = A_data
                B_k = B_data

            # A_k and B_k are numpy arrays
            # Construct Matrix objects for multiplication: Expr.mul(Matrix, Var)
            A_mat = Matrix.dense(A_k.shape[0], A_k.shape[1], A_k.flatten().tolist())
            B_mat = Matrix.dense(B_k.shape[0], B_k.shape[1], B_k.flatten().tolist())

            lhs = Expr.sub(mu_vars[k+1], Expr.add(Expr.mul(A_mat, mu_vars[k]), Expr.mul(B_mat, v_vars[k])))
            # enforce equality
            M.constraint(f"dynamics_{k}", lhs, Domain.equalsTo(np.zeros(nx)))

        # boundary constraints:
        M.constraint("mu1_eq", Expr.sub(mu_vars[0], mu0.tolist()), Domain.equalsTo(np.zeros(nx)))
        M.constraint("muN_eq", Expr.sub(mu_vars[-1], muf.tolist()), Domain.equalsTo(np.zeros(nx)))

        # waypoint constraints (assume waypoint on first two states (x,y) like MATLAB mu{wp_n(i)}(1:2) == wp(:,i))
        for idx, wpn in enumerate(wp_n):
            k = wpn - 1  # MATLAB 1-based -> Python 0-based
            if k < 0 or k >= N:
                raise IndexError(f"Waypoint index {wpn} out of range for N={N}")
            # set first two states equal
            # Build vector for difference and equal to zero
            mu_k = mu_vars[k]
            # create selector matrix to pick first 2 states
            sel = np.zeros((2, nx))
            sel[0,0] = 1.0
            sel[1,1] = 1.0
            Sel_mat = Matrix.dense(2, nx, sel.flatten().tolist())
            M.constraint(f"wp_{idx}", Expr.sub(Expr.mul(Sel_mat, mu_k),
                                               Matrix.dense(2,1, wp[:,idx].flatten().tolist())),
                         Domain.equalsTo(np.zeros(2)))

        # objective: sum mu_k' Q mu_k + v_k' R v_k
        # For each quadratic we use Expr.dot(var, Expr.mul(Matrix, var))
        obj_terms = []
        Q_mat = Matrix.dense(Q.shape[0], Q.shape[1], Q.flatten().tolist())
        R_mat = Matrix.dense(R.shape[0], R.shape[1], R.flatten().tolist())
        for k in range(N-1):
            # mu_k' Q mu_k
            obj_terms.append(Expr.dot(mu_vars[k], Expr.mul(Q_mat, mu_vars[k])))
            # v_k' R v_k
            obj_terms.append(Expr.dot(v_vars[k], Expr.mul(R_mat, v_vars[k])))
        # plus last mu_N term (k=N-1 in MATLAB loop included mu{N}? No, MATLAB summed i = 1:(N-1) only.
        # To mimic MATLAB, we do not include mu_N term unless you want to include endpoint cost.
        total = obj_terms[0] if len(obj_terms) == 1 else Expr.add(*obj_terms)
        M.objective(ObjectiveSense.Minimize, total)

        # solve
        M.setSolverParam("MSK_DPAR_INTPNT_TOL_REL_GAP", 1e-8)  # example param
        M.solve()

        # extract solution
        x_traj = np.zeros((nx, N))
        u_traj = np.zeros((nu, N-1))
        for k in range(N):
            x_traj[:,k] = mu_vars[k].level()
        for k in range(N-1):
            u_traj[:,k] = v_vars[k].level()

    return x_traj, u_traj

# Now call solveOMS on the linear plant
plant_lin['nu'] = 3
plant_lin['nx'] = 9
plant_lin['A'] = A_lin
plant_lin['B'] = B_lin

print("Solving OMS with MOSEK Fusion (this may take a second)...")
x_traj, u_traj = solveOMS_fusion(plant_lin, OMSparams, verbose=False)
print("Done. Trajectory shape:", x_traj.shape, "Control shape:", u_traj.shape)

# ---------------------------
# Differential flatness mapping (convert states to attitude and rotor commands)
# ---------------------------
# MATLAB:
# pos = x(1:3,:); vel = x(4:6,:); acc = x(7:9,:); jerk = u_lin
# t = acc + [0;0;g]; etc.

pos = x_traj[0:3, :]
vel = x_traj[3:6, :]
acc = x_traj[6:9, :]
jerk = u_traj

t_vec = acc + np.vstack([np.zeros((2, acc.shape[1])), np.ones((1,acc.shape[1]))*g])

# assume constant heading x_c = [1;0;0]
x_c = np.array([1.0, 0.0, 0.0])

# prepare rotation matrices
R_list = []
ea = np.zeros((3, acc.shape[1]))  # euler angles ZYX
x_b = np.zeros_like(t_vec)
y_b = np.zeros_like(t_vec)
z_b = np.zeros_like(t_vec)

for i in range(acc.shape[1]):
    t_i = t_vec[:,i]
    norm_t = np.linalg.norm(t_i)
    if norm_t < 1e-8:
        z_b_i = np.array([0, 0, 1.0])
    else:
        z_b_i = t_i / norm_t
    cross_y = np.cross(z_b_i, x_c)
    if np.linalg.norm(cross_y) < 1e-9:
        # x_c collinear with z_b; choose another axis
        x_c_alt = np.array([0.0, 1.0, 0.0])
        cross_y = np.cross(z_b_i, x_c_alt)
    y_b_i = cross_y / np.linalg.norm(cross_y)
    x_b_i = np.cross(y_b_i, z_b_i)
    R_i = np.column_stack([x_b_i, y_b_i, z_b_i])
    R_list.append(R_i)
    # convert to ZYX Euler
    ea[:, i] = R.from_matrix(R_i).as_euler('zyx', degrees=False)

# Tref = [pos; vel; ea]
Tref = np.vstack([pos, vel, ea])

# compute control mapping (motor commands u = [tau;p;q;r])
u = np.zeros((4, acc.shape[1]-1))
for i in range(acc.shape[1]-1):
    tau = np.linalg.norm(t_vec[:,i])
    zb = z_b[:,i]
    jerk_i = jerk[:,i]
    # h = m/tau*(jerk - dot(z_b, jerk)*z_b)
    if tau < 1e-8:
        h_i = np.zeros((3,))
    else:
        h_i = (m / tau) * (jerk_i - np.dot(zb, jerk_i)*zb)
    p = -np.dot(h_i, y_b[:,i])
    q = np.dot(h_i, x_b[:,i])
    r = 0.0
    u[:,i] = np.array([tau, p, q, r])

# ---------------------------
# Build linearized system matrices A[i], B[i], D[i] for each step
# ---------------------------
plant['nu'] = 4   # After flat mapping
plant['A'] = [None]*(N-1)
plant['B'] = [None]*(N-1)
plant['D'] = [None]*(N-1)

for i in range(N-1):
    # user must implement eval_A/B/D
    try:
        A_i = eval_A(Tref[:,i], u[:,i], plant)
        B_i = eval_B(Tref[:,i], u[:,i], plant)
        D_i = eval_D(Tref[:,i], u[:,i], plant)
    except NotImplementedError:
        # As fallback we approximate with identity + small noise for illustration.
        A_i = np.eye(9)
        B_i = np.zeros((9,4))
        D_i = np.zeros((9,9))
    # Add the same extra terms you had in MATLAB:
    # + blkdiag(40*DT^2*eye(3), 3*DT*eye(3), 5*DT*eye(3))
    extra = np.block([
        [40*(DT**2)*np.eye(3), np.zeros((3,6))],
        [np.zeros((3,3)), 3*DT*np.eye(3), np.zeros((3,3))],
        [np.zeros((3,6)), 5*DT*np.eye(3)]
    ])
    # careful shape of extra: ensure 9x9
    if extra.shape != (9,9):
        extra = np.zeros((9,9))
    D_i = D_i + extra
    plant['A'][i] = A_i
    plant['B'][i] = B_i
    plant['D'][i] = D_i

# ---------------------------
# Build OCSparams and call solveOCS_LTV (placeholder)
# ---------------------------
OCSparams = {}
OCSparams['N'] = N
OCSparams['Q'] = np.block([ [10*np.eye(3), np.zeros((3,6))],
                            [np.zeros((3,3)), 10*np.eye(3), np.zeros((3,3))],
                            [np.zeros((3,6)), 10*np.eye(3)] ])  # This block assembly isn't exact; replace with diag
OCSparams['R'] = 0.1*np.eye(4)
OCSparams['Si'] = np.block([ [0.03*np.eye(3), np.zeros((3,6))],
                             [np.zeros((3,3)), 0.001*np.eye(3), np.zeros((3,3))],
                             [np.zeros((3,6)), 0.001*np.eye(3)] ])
OCSparams['Sf'] = np.block([ [0.0005*np.eye(3), np.zeros((3,6))],
                             [np.zeros((3,3)), 0.01*np.eye(3), np.zeros((3,3))],
                             [np.zeros((3,6)), 0.01*np.eye(3)] ])


Solving OMS with MOSEK Fusion (this may take a second)...


DimensionError: Conflicting shape definitions